# Track 2 · Stage 2 — Synthesize the code-mixed **audio**

Pulls the ~16k sentences that `01a` pushed, voices them with **Indic Parler-TTS**
(Rohit ♂ / Divya ♀, 50/50), and pushes ~22h of 16 kHz audio to
`RohanRamesh/hi-en-synth-cs`.

**Separate notebook because `parler-tts` hard-pins `transformers==4.46.1`**, which cannot
load the Gemma 3 used in `01a`. No LLM is loaded here.

### Before you run
* **Accept the licence for [`ai4bharat/indic-parler-tts`](https://huggingface.co/ai4bharat/indic-parler-tts)** — it is gated and has no
  ungated mirror. If your HF token is *fine-grained*, also tick **"Read access to contents
  of all public gated repos you can access"**; accepting the licence alone is not enough.
* HF **write** token in Kaggle Secrets as `HF_TOKEN`. Internet **on**, **GPU T4 ×2**.

Runtime ≈ 4–6h. Sharded — re-running skips every clip already on disk, so a 12h timeout
costs at most the in-flight shard.

## 0 · Install

**Cell order matters.** The `transformers` downgrade silently no-ops if anything has already imported `transformers`. Do not import it above this cell.

In [ ]:
!pip install -q "transformers==4.46.1" "datasets<4" accelerate soxr librosa soundfile
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q --force-reinstall --no-deps git+https://github.com/BRUH-MAIN/codeswitching.git

# This NOTEBOOK's own version. `pip install` updates the csasr PACKAGE but NOT the
# .ipynb -- an old notebook against a new package is a real and confusing failure.
NOTEBOOK_VERSION = "0.11.1"

import csasr, transformers
assert csasr.__version__ == NOTEBOOK_VERSION, (
    f"csasr package is {csasr.__version__} but this NOTEBOOK is {NOTEBOOK_VERSION}.\n"
    "  package older  -> restart the kernel (Run > Restart & clear); pip skips a\n"
    "                    reinstall when the version looks satisfied.\n"
    "  notebook older -> re-download it from the repo; pip does NOT update .ipynb files."
)
assert transformers.__version__ == "4.46.1", (
    f"transformers is {transformers.__version__}, expected 4.46.1 - something imported "
    "it before the downgrade took effect. Restart the kernel."
)
print("csasr", csasr.__version__, "| transformers", transformers.__version__)

In [ ]:
import os, subprocess, sys, itertools
from pathlib import Path

os.environ["HF_HOME"] = "/kaggle/temp/hf"
Path(os.environ["HF_HOME"]).mkdir(parents=True, exist_ok=True)

# NEVER let transformers import TensorFlow. `parler_tts` imports
# `transformers.PreTrainedModel`, which reaches image_transforms.py and runs
# `if is_tf_available(): import tensorflow`. Kaggle HAS TensorFlow, but it wants a
# newer protobuf than our pins leave behind, so it dies with
#   ImportError: cannot import name 'runtime_version' from 'google.protobuf'
# We never use TF. USE_TF=0 makes is_tf_available() False and the import vanishes.
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])
HF_TOKEN = os.environ["HF_TOKEN"]

import torch
N_GPU = max(1, torch.cuda.device_count())

SYNTH_REPO = "RohanRamesh/hi-en-synth-cs"
TTS        = "ai4bharat/indic-parler-tts"
NUM_SHARDS = 4          # finer than N_GPU on purpose: a timeout costs one shard

WORK  = Path("/kaggle/working")
MAN   = WORK / "manifests"; MAN.mkdir(parents=True, exist_ok=True)
AUDIO = WORK / "audio";     AUDIO.mkdir(parents=True, exist_ok=True)

def run(*args):
    print(">", " ".join(str(a) for a in args), flush=True)
    p = subprocess.run([sys.executable, "-m", *args])
    if p.returncode != 0:
        raise RuntimeError(
            f"{args[0]} failed (exit {p.returncode}). The real error is printed ABOVE "
            f"this traceback - scroll up in this cell's output."
        )

def run_parallel(cmds):
    """One process per GPU, pinned with CUDA_VISIBLE_DEVICES; wait for all.

    Running the shards SEQUENTIALLY leaves Kaggle's second T4 completely idle --
    the same waste already fixed in 01a and 03_eval.
    """
    procs = []
    for i, cmd in enumerate(cmds):
        gpu = i % N_GPU
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        print(f"> GPU{gpu}: {' '.join(str(c) for c in cmd)}", flush=True)
        procs.append(subprocess.Popen(
            [sys.executable, "-m", *[str(c) for c in cmd]], env=env))
    for i, p in enumerate(procs):
        if p.wait() != 0:
            raise RuntimeError(f"shard {i} failed (exit {p.returncode}). The real error "
                               "is printed ABOVE - scroll up in this cell's output.")

print(f"{N_GPU} GPU(s), {NUM_SHARDS} shards")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0b · PREFLIGHT — Parler-TTS is gated

Fails in seconds rather than after a 6-hour run has started.

In [ ]:
from huggingface_hub import model_info
from huggingface_hub.utils import GatedRepoError, RepositoryNotFoundError

try:
    model_info(TTS, token=HF_TOKEN)
    print(f"  OK  {TTS}")
except GatedRepoError:
    raise SystemExit(
        f"PREFLIGHT FAILED: {TTS} is gated.\n"
        f"  1. Accept the licence at https://huggingface.co/{TTS}\n"
        f"  2. If your HF token is FINE-GRAINED, also tick 'Read access to contents of\n"
        f"     all public gated repos you can access', then update the Kaggle Secret."
    )
except RepositoryNotFoundError:
    raise SystemExit(f"PREFLIGHT FAILED: {TTS} not found, or your token cannot see it.")

## 1 · Pull the sentences generated by `01a`

In [ ]:
from datasets import load_dataset
from csasr.manifest import read_jsonl, write_jsonl

ds = load_dataset(SYNTH_REPO, "sentences", split="train", token=HF_TOKEN)
write_jsonl(MAN / "sentences.jsonl", [dict(r) for r in ds])
print(f"{len(ds):,} sentences to synthesize")
print(ds[0]["text"])

## 2 · Synthesize

Indic Parler-TTS emits **44.1 kHz** → resampled to 16 kHz and written as **int16**
(`synthesize.py` reads `model.config.sampling_rate`; it never hardcodes the rate).

**Both GPUs, in parallel.** Shards run two at a time, one per T4 — running them
sequentially would leave half the machine idle, which is exactly the waste already fixed in
01a and 03_eval.

**Resumable.** Every clip already on disk is skipped, so re-running this cell after a
timeout costs only the in-flight batch.

In [ ]:
BATCH = 16     # Parler is small (938M); 8 left the GPU underfed

# Two shards at a time -- one per GPU.
for start in range(0, NUM_SHARDS, N_GPU):
    group = range(start, min(start + N_GPU, NUM_SHARDS))
    run_parallel([
        ["csasr.tts.synthesize",
         "--sentences", MAN / "sentences.jsonl",
         "--audio-dir", AUDIO,
         "--out", MAN / f"train_t2.shard{s}.jsonl",
         "--shard", s, "--num-shards", NUM_SHARDS,
         "--batch-size", BATCH]
        for s in group
    ])

## 3 · Merge shards, build Train_T1 by reference

### GATE 2 — `Train_T1 ⊆ Train_T2`, durations ≈ 8h / 22h

In [ ]:
merged = list(itertools.chain.from_iterable(
    read_jsonl(MAN / f"train_t2.shard{i}.jsonl") for i in range(NUM_SHARDS)
))
write_jsonl(MAN / "train_t2.jsonl", merged)
hours = sum(r["dur"] for r in merged) / 3600
print(f"Train_T2: {len(merged):,} clips, {hours:.2f} h (paper: 22 h)")

run("csasr.tts.make_subset", "--t2", MAN / "train_t2.jsonl",
    "--out", WORK / "t1_ids.json", "--hours", "8.0")

### Listen to it. Two clips per voice — do not skip this.

In [ ]:
import IPython.display as ipd, random
from csasr.tts.speakers import assign_speaker
random.seed(0)
for voice in ("Rohit", "Divya"):
    picks = [r for r in merged if assign_speaker(r["sent_id"]).name == voice][:200]
    for r in random.sample(picks, 2):
        print(f"[{voice}] {r['text']}")
        ipd.display(ipd.Audio(r["wav"]))

## 4 · Push the synthetic corpus

Parquet + FLAC ≈ 1.4 GB (vs 2.5 GB as WAV). The round-trip check catches a corrupted upload *before* a 3h training run.

In [ ]:
run("csasr.data.push_to_hub", "--manifest", MAN / "train_t2.jsonl",
    "--repo", SYNTH_REPO, "--config", "synth_t2", "--verify")

from huggingface_hub import HfApi
HfApi().upload_file(path_or_fileobj=str(WORK / "t1_ids.json"),
                    path_in_repo="t1_ids.json", repo_id=SYNTH_REPO,
                    repo_type="dataset", token=HF_TOKEN)
print("done ->", SYNTH_REPO, "\nnext: 02_train.ipynb")